# News Discovery System — Colab Gradio Launcher (with Geospatial Map)

This notebook runs the repository workflow end-to-end in **Google Colab** and launches a **Gradio UI** with inspectable stage outputs and geospatial visualization.

## 1) Environment setup
- Detect Colab runtime
- Define project directory
- Set plotting backend for notebook runtime stability

In [ ]:
from pathlib import Path
import os
import platform

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False
PROJECT_DIR = Path('/content/news-discovery-system') if IN_COLAB else Path.cwd()
os.environ['MPLBACKEND'] = 'Agg'

print('Python:', platform.python_version())
print('In Colab:', IN_COLAB)
print('Project directory:', PROJECT_DIR)

## 2) Dependency install
Installs notebook and app dependencies including geospatial libraries used by the map stage.

In [ ]:
!pip -q install --upgrade pip
!pip -q install gradio matplotlib folium geopy geotext branca

## 3) Project load
If the repo is not already present in `/content`, this cell clones it.

> Set `REPO_URL` to your repository URL before first run in a clean Colab runtime.

In [ ]:
import shutil

REPO_URL = 'https://github.com/<YOUR_ORG>/news-discovery-system.git'

if (PROJECT_DIR / 'gr_app.py').exists():
    print('Repository already available:', PROJECT_DIR)
else:
    if '<YOUR_ORG>' in REPO_URL:
        raise ValueError('Set REPO_URL to your real GitHub repository URL, then rerun this cell.')

    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)

    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}

## 4) Optional patching
This is a non-destructive runtime patch step:
- Verifies required files
- Adds repo root to `sys.path`
- Validates workflow import

In [ ]:
import sys

required = [
    PROJECT_DIR / 'src' / 'news_app' / 'workflow.py',
    PROJECT_DIR / 'config' / 'sources.json',
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.news_app.workflow import RunInput, run_workflow
print('Workflow import check: OK')

## 5) Launch Gradio UI (share=True)
Inputs: topic, start_date, end_date.

Outputs: ingestion JSON, normalization JSON, aggregation JSON, timeline plot, geospatial map.

In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from functools import lru_cache
from typing import Any

import folium
import gradio as gr
import matplotlib.pyplot as plt
from geopy.exc import GeocoderServiceError
from geopy.geocoders import Nominatim
from geotext import GeoText


def _validate_inputs(topic: str, start_date: str, end_date: str):
    topic = str(topic or '').strip()
    if not topic:
        raise ValueError('Topic is required.')

    start = datetime.strptime(str(start_date).strip(), '%Y-%m-%d').date()
    end = datetime.strptime(str(end_date).strip(), '%Y-%m-%d').date()

    if start > end:
        raise ValueError('Start date must be on or before end date.')
    if (end - start).days > 30:
        raise ValueError('Date range must be 30 days or less.')
    if end > datetime.now(timezone.utc).date():
        raise ValueError('End date cannot be in the future.')

    return topic, start, end


def _timeline_figure(daily_counts: list[dict[str, Any]]):
    fig, ax = plt.subplots(figsize=(10, 4))
    if not daily_counts:
        ax.text(0.5, 0.5, 'No timeline data returned.', ha='center', va='center')
        ax.set_title('Daily Article Counts')
        ax.set_xlabel('Date')
        ax.set_ylabel('Articles')
        fig.tight_layout()
        return fig

    days = [point['day'] for point in daily_counts]
    counts = [point['article_count'] for point in daily_counts]
    ax.plot(days, counts, marker='o')
    ax.set_title('Daily Article Counts')
    ax.set_xlabel('Date')
    ax.set_ylabel('Articles')
    ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    return fig


def _pretty_json(data: Any) -> str:
    return json.dumps(data, indent=2, ensure_ascii=False, default=str)


@lru_cache(maxsize=1024)
def _geocode_place(name: str):
    geolocator = Nominatim(user_agent='news_discovery_colab')
    try:
        loc = geolocator.geocode(name, timeout=10)
    except GeocoderServiceError:
        return None

    if not loc:
        return None

    return {'display_name': loc.address, 'latitude': float(loc.latitude), 'longitude': float(loc.longitude)}


def _extract_geospatial_entities(canonical_articles: list[dict[str, Any]]):
    entities = []
    for article in canonical_articles:
        title = article.get('title') or ''
        places = GeoText(title)
        candidates = list(dict.fromkeys((places.cities or []) + (places.countries or [])))
        for idx, place in enumerate(candidates):
            geo = _geocode_place(place)
            if not geo:
                continue
            is_city = place in (places.cities or [])
            confidence = 0.82 if is_city else 0.62
            ambiguity = len(candidates) > 1
            entities.append({
                'location_id': f"{article['article_id']}:loc:{idx}",
                'article_id': article['article_id'],
                'title': article.get('title'),
                'article_url': article.get('url'),
                'place_label': place,
                'resolved_name': geo['display_name'],
                'latitude': geo['latitude'],
                'longitude': geo['longitude'],
                'confidence_score': confidence,
                'extraction_method': 'explicit',
                'evidence_text_span': place,
                'ambiguity_flag': ambiguity,
                'ambiguity_notes': 'Multiple location candidates in title.' if ambiguity else '',
                'cluster_ids': article.get('cluster_ids', []),
                'created_at': datetime.now(timezone.utc).isoformat(),
            })
    return entities


def _aggregate_markers(entities: list[dict[str, Any]]):
    grouped = defaultdict(list)
    for entity in entities:
        key = (round(entity['latitude'], 2), round(entity['longitude'], 2), entity['resolved_name'])
        grouped[key].append(entity)

    markers = []
    for (lat, lon, resolved_name), group in grouped.items():
        article_ids = sorted({item['article_id'] for item in group})
        article_links = sorted({(item.get('title') or item['article_id'], item.get('article_url') or '') for item in group})
        cluster_ids = sorted({cid for item in group for cid in (item.get('cluster_ids') or [])})
        avg_confidence = sum(item['confidence_score'] for item in group) / len(group)
        ambiguity_count = sum(1 for item in group if item['ambiguity_flag'])
        intensity = 'high' if len(article_ids) >= 5 else 'medium' if len(article_ids) >= 2 else 'low'

        markers.append({
            'marker_id': f"marker_{abs(hash((lat, lon, resolved_name))) % 10**8}",
            'resolved_name': resolved_name,
            'latitude': lat,
            'longitude': lon,
            'unique_article_count': len(article_ids),
            'mention_count': len(group),
            'avg_confidence': round(avg_confidence, 3),
            'uncertainty_score': round(1 - avg_confidence, 3),
            'ambiguity_count': ambiguity_count,
            'intensity_band': intensity,
            'article_ids': article_ids,
            'article_links': [{'label': label, 'url': url} for label, url in article_links],
            'cluster_ids': cluster_ids,
        })

    return sorted(markers, key=lambda x: x['unique_article_count'], reverse=True)


def _map_html(markers: list[dict[str, Any]]) -> str:
    if not markers:
        return '<div style="padding:12px;border:1px solid #ddd;border-radius:8px;">No geospatial markers were extracted from current results.</div>'

    center_lat = sum(m['latitude'] for m in markers) / len(markers)
    center_lon = sum(m['longitude'] for m in markers) / len(markers)
    fmap = folium.Map(location=[center_lat, center_lon], zoom_start=2, tiles='CartoDB positron')
    color_by_intensity = {'low': '#2a9d8f', 'medium': '#f4a261', 'high': '#e76f51'}

    for marker in markers:
        color = color_by_intensity.get(marker['intensity_band'], '#577590')
        radius = 6 + min(marker['unique_article_count'], 15)
        weight = 4 if marker['ambiguity_count'] > 0 else 1

        article_html = []
        for item in marker['article_links'][:8]:
            if item['url']:
                article_html.append(f"<li><a href='{item['url']}' target='_blank'>{item['label']}</a></li>")
            else:
                article_html.append(f"<li>{item['label']}</li>")

        cluster_text = ', '.join(marker['cluster_ids']) if marker['cluster_ids'] else 'None'
        popup_html = (
            f"<div style='width:360px;'><b>{marker['resolved_name']}</b><br/>"
            f"Intensity: {marker['intensity_band']}<br/>"
            f"Unique articles: {marker['unique_article_count']}<br/>"
            f"Avg confidence: {marker['avg_confidence']}<br/>"
            f"Uncertainty: {marker['uncertainty_score']}<br/>"
            f"Ambiguity count: {marker['ambiguity_count']}<br/>"
            f"Clusters: {cluster_text}<hr/><b>Articles</b><ul>{''.join(article_html) if article_html else '<li>No article URL</li>'}</ul></div>"
        )

        folium.CircleMarker(
            location=[marker['latitude'], marker['longitude']],
            radius=radius,
            color='#000000' if marker['ambiguity_count'] > 0 else color,
            fill=True,
            fill_color=color,
            fill_opacity=max(0.35, 1 - marker['uncertainty_score']),
            weight=weight,
            popup=folium.Popup(popup_html, max_width=420),
            tooltip=f"{marker['resolved_name']} | articles={marker['unique_article_count']} | uncertainty={marker['uncertainty_score']}",
        ).add_to(fmap)

    legend_html = "<div style='position: fixed; bottom: 24px; left: 24px; z-index:9999; background:white; border:1px solid #ddd; padding:10px; font-size:12px;'><b>Legend</b><br/><span style='color:#2a9d8f;'>●</span> Low intensity (1 article)<br/><span style='color:#f4a261;'>●</span> Medium intensity (2-4 articles)<br/><span style='color:#e76f51;'>●</span> High intensity (5+ articles)<br/>Black outline = ambiguity present</div>"
    fmap.get_root().html.add_child(folium.Element(legend_html))
    return fmap._repr_html_()


def run_ui_workflow(topic: str, start_date: str, end_date: str):
    try:
        topic, start, end = _validate_inputs(topic, start_date, end_date)
        result = run_workflow(RunInput(topic=topic, start_date=start, end_date=end))
        stages = result.get('stages', {})
        ingestion = stages.get('ingestion', {})
        normalization = stages.get('normalization', {})
        aggregation = stages.get('aggregation', {})

        canonical_articles = normalization.get('canonical_articles', [])
        entities = _extract_geospatial_entities(canonical_articles)
        markers = _aggregate_markers(entities)

        aggregation_with_geo = {
            **aggregation,
            'geospatial': {
                'location_entities': entities,
                'map_markers': markers,
                'marker_count': len(markers),
            },
        }

        return (
            f"Completed: {result.get('run_id', 'unknown-run-id')}",
            _pretty_json(ingestion),
            _pretty_json(normalization),
            _pretty_json(aggregation_with_geo),
            _timeline_figure(aggregation.get('daily_counts', [])),
            _map_html(markers),
        )
    except Exception as exc:
        return (
            f'Workflow execution failed: {exc}',
            '{}',
            '{}',
            '{}',
            _timeline_figure([]),
            '<div style="padding:12px;color:#b00020;">Map unavailable due to workflow error.</div>',
        )


def build_colab_app():
    today = datetime.now(timezone.utc).date()
    default_end = today
    default_start = today - timedelta(days=7)

    with gr.Blocks(title='News Discovery Colab UI') as demo:
        gr.Markdown('News Discovery Colab UI: ingestion -> normalization -> aggregation -> timeline -> geospatial map')
        with gr.Row():
            topic = gr.Textbox(label='topic', placeholder='e.g., semiconductor', scale=2)
            start_date = gr.Textbox(label='start_date (YYYY-MM-DD)', value=str(default_start))
            end_date = gr.Textbox(label='end_date (YYYY-MM-DD)', value=str(default_end))

        run_button = gr.Button('Run Workflow')
        status = gr.Textbox(label='status', interactive=False)
        ingestion_json = gr.Code(label='ingestion JSON', language='json')
        normalization_json = gr.Code(label='normalization JSON', language='json')
        aggregation_json = gr.Code(label='aggregation JSON', language='json')
        timeline_plot = gr.Plot(label='timeline plot')
        geo_map = gr.HTML(label='geospatial map')

        run_button.click(
            fn=run_ui_workflow,
            inputs=[topic, start_date, end_date],
            outputs=[status, ingestion_json, normalization_json, aggregation_json, timeline_plot, geo_map],
        )

    return demo


demo = build_colab_app()
demo.queue()
demo.launch(share=True, inline=False, debug=True)

## Validation checklist (analyst)
1. Run the workflow with topic + date range.
2. Verify **ingestion success** in `ingestion JSON` (`hits_count` > 0).
3. Verify **location extraction correctness** in `aggregation JSON.geospatial.location_entities`.
4. Verify **aggregation correctness** in `aggregation JSON.daily_counts` and `aggregation JSON.geospatial.map_markers`.
5. Verify **timeline accuracy** by matching plot points to `daily_counts`.
6. Verify **map correctness** using marker popup details: intensity, uncertainty, linked articles, and cluster IDs (if available).